# Global dataset merge: low-memory schema and key inspection

Run this before merging. It reads Parquet schema/row-group metadata without loading whole tables into pandas, takes tiny previews through DuckDB, reads only a few CSV rows, and inspects Zarr metadata without loading patch values. It writes `global_merge_schema_report.txt` and `.json` into `Data/Global`.

The output is evidence for the merge design; column names alone do not prove a key is unique or semantically correct.


In [1]:
from pathlib import Path
from datetime import datetime
import json, csv, os, sys, subprocess, importlib.util

for package, module in [("duckdb", "duckdb"), ("pyarrow", "pyarrow"), ("zarr", "zarr")]:
    if importlib.util.find_spec(module) is None:
        print("Installing missing dependency:", package)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

import duckdb
import pyarrow.parquet as pq
import zarr
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != "Global" or not (ROOT / "global_ismn_daily.parquet").exists():
    ROOT = Path.home() / "Code/UG_SM/Data/Global"
if not (ROOT / "global_ismn_daily.parquet").exists():
    raise FileNotFoundError(f"Could not locate Data/Global. Set ROOT manually. Current value: {ROOT}")

REPORT_JSON = ROOT / "global_merge_schema_report.json"
REPORT_TXT = ROOT / "global_merge_schema_report.txt"
print("Project:", ROOT.resolve())
print("DuckDB:", duckdb.__version__, "| PyArrow:", __import__("pyarrow").__version__, "| Zarr:", zarr.__version__)


Project: /home/mayur/Code/UG_SM/Data/Global
DuckDB: 1.5.6 | PyArrow: 23.0.1 | Zarr: 3.4.0


In [2]:
PARQUETS = [
    ROOT / "global_ismn_daily.parquet",
    ROOT / "station_date_candidates.parquet",
    ROOT / "stations.parquet",
    ROOT / "era5_backend/era5_by_sample.parquet",
    ROOT / "era5_backend/era5_point_index.parquet",
    ROOT / "era5_backend/era5_sample_features.parquet",
    ROOT / "smap/smap_by_sample.parquet",
    ROOT / "smap/station_smap_grid_mapping.parquet",
    ROOT / "sentinel1/clusters.parquet",
    ROOT / "sentinel1/stations_with_clusters.parquet",
    ROOT / "sentinel1/samples_2015.parquet",
    ROOT / "sentinel1/samples_2016.parquet",
]
CSVS = [ROOT / "Statics/global_static_soil.csv", ROOT / "Statics/global_static_terrain.csv"]
ZARR_ROOTS = [
    ROOT / "modis/evi_lst_patch_dataset",
    ROOT / "modis/modis_patch_dataset",
    ROOT / "modis/ndvi_lst_patch_dataset",
    ROOT / "sentinel1/nearest_valid_overview_v1",
]
report = {"created_at": datetime.now().isoformat(timespec="seconds"), "root": str(ROOT.resolve()), "parquet": [], "csv": [], "zarr": [], "key_inventory": []}
con = duckdb.connect(":memory:")
con.execute("SET memory_limit='512MB'")
con.execute("SET threads=2")
con.execute("SET TimeZone='UTC'")

def sqlq(x): return "'" + str(x).replace("'", "''") + "'"
def safe(v):
    if hasattr(v, "isoformat"): return v.isoformat()
    if hasattr(v, "item"):
        try: return v.item()
        except Exception: pass
    try:
        if v is None or pd.isna(v): return None
    except Exception: pass
    return str(v)

print("Found Parquet files:")
for p in PARQUETS:
    if p.exists(): print(" ", p.relative_to(ROOT), f"({p.stat().st_size:,} bytes)")
    else: print("  NOT FOUND:", p.relative_to(ROOT))


Found Parquet files:
  global_ismn_daily.parquet (60,840,090 bytes)
  station_date_candidates.parquet (5,883,140 bytes)
  stations.parquet (122,803 bytes)
  era5_backend/era5_by_sample.parquet (95,951,021 bytes)
  era5_backend/era5_point_index.parquet (18,047 bytes)
  era5_backend/era5_sample_features.parquet (96,042,457 bytes)
  smap/smap_by_sample.parquet (29,441,475 bytes)
  smap/station_smap_grid_mapping.parquet (239,531 bytes)
  sentinel1/clusters.parquet (59,035 bytes)
  sentinel1/stations_with_clusters.parquet (129,255 bytes)
  sentinel1/samples_2015.parquet (3,327,594 bytes)
  sentinel1/samples_2016.parquet (3,890,790 bytes)


In [3]:
# Parquet metadata + max-three-row preview. No full Parquet is loaded into pandas.
for path in PARQUETS:
    if not path.exists(): continue
    print("\n" + "="*88 + "\nPARQUET:", path.relative_to(ROOT))
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    cols = [{"name": f.name, "type": str(f.type), "nullable": f.nullable} for f in schema]
    item = {"path": str(path.relative_to(ROOT)), "size_bytes": path.stat().st_size,
            "rows": pf.metadata.num_rows, "row_groups": pf.metadata.num_row_groups, "columns": cols}
    print(f"Rows={item['rows']:,}; row groups={item['row_groups']}; size={item['size_bytes']:,} bytes")
    for f in cols: print(f"  {f['name']}: {f['type']}")
    preview_cols = [f["name"] for f in cols[:12]]
    quoted = ", ".join('"' + c.replace('"','""') + '"' for c in preview_cols)
    try:
        df = con.execute(f"SELECT {quoted} FROM read_parquet({sqlq(path.resolve())}) LIMIT 3").fetchdf()
        display(df)
        item["preview_columns"] = preview_cols
        item["preview_rows"] = [{k: safe(v) for k,v in row.items()} for row in df.to_dict(orient="records")]
    except Exception as e:
        item["preview_error"] = repr(e)
        print("Preview query failed:", repr(e))
    report["parquet"].append(item)



PARQUET: global_ismn_daily.parquet
Rows=5,045,849; row groups=4284; size=60,840,090 bytes
  station_id: string
  network: string
  network_name: string
  station: string
  latitude: double
  longitude: double
  elevation_m: double
  sensor: string
  depth_from_m: double
  depth_to_m: double
  depth_class: string
  sample_date: date32[day]
  soil_moisture_m3_m3: float
  observation_count: int32
  source_file: string
  clay_pct: float
  sand_pct: float
  silt_pct: float
  climate_zone: string


,station_id,network,network_name,station,latitude,longitude,elevation_m,sensor,depth_from_m,depth_to_m,depth_class,sample_date
0,AMMA-CATCH__Banizoumbou,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,Campbell-CS616,0.05,0.05,near_surface,2015-04-01
1,AMMA-CATCH__Banizoumbou,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,Campbell-CS616,0.05,0.05,near_surface,2015-04-02
2,AMMA-CATCH__Banizoumbou,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,Campbell-CS616,0.05,0.05,near_surface,2015-04-03



PARQUET: station_date_candidates.parquet
Rows=3,514,643; row groups=29; size=5,883,140 bytes
  sample_id: int64
  station_id: string
  sample_date: date32[day]
  network: string
  network_name: string
  station: string
  latitude: double
  longitude: double
  elevation_m: double
  climate_zone: string
  source_row_count: int64
  min_depth_from_m: double
  max_depth_to_m: double
  sensors: string
  depth_classes: string


,sample_id,station_id,sample_date,network,network_name,station,latitude,longitude,elevation_m,climate_zone,source_row_count,min_depth_from_m
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05



PARQUET: stations.parquet
Rows=2,210; row groups=1; size=122,803 bytes
  station_index: int32
  station_id: string
  network: string
  network_name: string
  station: string
  latitude: double
  longitude: double
  elevation_m: double
  climate_zone: string


,station_index,station_id,network,network_name,station,latitude,longitude,elevation_m,climate_zone
0,0,AMMA-CATCH__Banizoumbou,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,BSh
1,1,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,Aw
2,2,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,AMMA-CATCH,Belefoungou-Top,9.78986,1.70994,414.0,Aw


... (14 outputs left out)


PARQUET: sentinel1/samples_2015.parquet
Rows=329,903; row groups=1; size=3,327,594 bytes
  sample_id: int64
  station_id: string
  sample_date: timestamp[ns]
  cluster_id: int64
  year: int32
  row_index: int64


,sample_id,station_id,sample_date,cluster_id,year,row_index
0,549114,KIHS_CMC__Cheongmicheon_A1,2015-04-01,0,2015,0
1,549115,KIHS_CMC__Cheongmicheon_A1,2015-04-02,0,2015,1
2,549116,KIHS_CMC__Cheongmicheon_A1,2015-04-03,0,2015,2



PARQUET: sentinel1/samples_2016.parquet
Rows=394,790; row groups=1; size=3,890,790 bytes
  sample_id: int64
  station_id: string
  sample_date: timestamp[ns]
  cluster_id: int64
  year: int32
  row_index: int64


,sample_id,station_id,sample_date,cluster_id,year,row_index
0,549361,KIHS_CMC__Cheongmicheon_A1,2016-03-23,0,2016,0
1,549362,KIHS_CMC__Cheongmicheon_A1,2016-03-24,0,2016,1
2,549363,KIHS_CMC__Cheongmicheon_A1,2016-03-25,0,2016,2


In [4]:
# CSV header and three-record preview only.
for path in CSVS:
    print("\n" + "="*88 + "\nCSV:", path.relative_to(ROOT))
    if not path.exists():
        report["csv"].append({"path": str(path.relative_to(ROOT)), "error": "not found"})
        print("NOT FOUND")
        continue
    with path.open("r", newline="", encoding="utf-8-sig", errors="replace") as f:
        reader = csv.reader(f)
        headers = next(reader, [])
        rows = [r for _, r in zip(range(3), reader)]
    print("Bytes:", f"{path.stat().st_size:,}")
    print("Columns:", headers)
    for row in rows: print(" ", row)
    report["csv"].append({"path": str(path.relative_to(ROOT)), "size_bytes": path.stat().st_size, "columns": headers, "preview_rows": rows})



CSV: Statics/global_static_soil.csv
Bytes: 147,419
Columns: ['station_key', 'station_id', 'network', 'clay_pct', 'sand_pct', 'silt_pct']
  ['AMMA-CATCH::AMMA-CATCH__Banizoumbou', 'AMMA-CATCH__Banizoumbou', 'AMMA-CATCH', '6.9', '83.0', '10.1']
  ['AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid', 'AMMA-CATCH__Belefoungou-Mid', 'AMMA-CATCH', '13.7', '68.5', '17.7']
  ['AMMA-CATCH::AMMA-CATCH__Belefoungou-Top', 'AMMA-CATCH__Belefoungou-Top', 'AMMA-CATCH', '14.7', '64.7', '20.6']

CSV: Statics/global_static_terrain.csv
Bytes: 183,905
Columns: ['station_key', 'station_id', 'network', 'elevation_m', 'slope_deg', 'aspect_deg']
  ['AMMA-CATCH::AMMA-CATCH__Banizoumbou', 'AMMA-CATCH__Banizoumbou', 'AMMA-CATCH', '208.64137', '1.3030396', '351.19223']
  ['AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid', 'AMMA-CATCH__Belefoungou-Mid', 'AMMA-CATCH', '417.68732', '3.5743034', '88.47684']
  ['AMMA-CATCH::AMMA-CATCH__Belefoungou-Top', 'AMMA-CATCH__Belefoungou-Top', 'AMMA-CATCH', '422.6819', '0.9841093', '286.3957']


In [5]:
# Zarr metadata only. Finds a Zarr root at the folder, its direct children, or one level deeper.
def zarr_root(p): return p.is_dir() and any((p / m).exists() for m in ("zarr.json", ".zgroup", ".zarray"))
def attrs_dict(obj):
    try:
        try: return dict(obj.attrs.asdict())
        except Exception: return dict(obj.attrs)
    except Exception: return {}
def inspect_store(p):
    item = {"path": str(p.relative_to(ROOT)), "attrs": {}, "arrays": [], "error": None}
    try:
        group = zarr.open_group(str(p), mode="r")
        item["attrs"] = attrs_dict(group)
        def visit(name, obj):
            if hasattr(obj, "shape") and hasattr(obj, "dtype"):
                item["arrays"].append({"name": name, "shape": list(obj.shape), "dtype": str(obj.dtype),
                    "chunks": list(obj.chunks) if getattr(obj, "chunks", None) is not None else None,
                    "attrs": attrs_dict(obj)})
        try: group.visititems(visit)
        except Exception as ex: item["walk_warning"] = repr(ex)
        print("\nZARR:", item["path"], "attrs=", item["attrs"])
        print("Arrays:", len(item["arrays"]))
        for a in item["arrays"]: print(f"  {a['name']}: shape={a['shape']} dtype={a['dtype']} chunks={a['chunks']}")
    except Exception as e:
        item["error"] = repr(e)
        print("Zarr open failed:", p, repr(e))
    return item

for folder in ZARR_ROOTS:
    print("\n" + "="*88 + "\nZARR FOLDER:", folder.relative_to(ROOT))
    if not folder.exists():
        report["zarr"].append({"path": str(folder.relative_to(ROOT)), "error": "not found"})
        print("NOT FOUND")
        continue
    if zarr_root(folder):
        stores = [folder]
    else:
        children = sorted([p for p in folder.iterdir() if p.is_dir()])
        stores = [p for p in children if zarr_root(p)]
        if not stores:
            for child in children:
                try: stores.extend(p for p in child.iterdir() if p.is_dir() and zarr_root(p))
                except OSError: pass
    if not stores:
        entry = {"path": str(folder.relative_to(ROOT)), "error": "No Zarr root detected within two directory levels",
                 "children": [p.name for p in sorted(folder.iterdir())[:60]]}
        report["zarr"].append(entry)
        print(entry["error"], "| children:", entry["children"])
    else:
        print("Stores found:", len(stores))
        for store in stores: report["zarr"].append(inspect_store(store))



ZARR FOLDER: modis/evi_lst_patch_dataset
Stores found: 12

ZARR: modis/evi_lst_patch_dataset/2015/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2016/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2017/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2018/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2019/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2020/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2021/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2022/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2023/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2024/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2025/patches.zarr attrs= {}
Arrays: 0

ZARR: modis/evi_lst_patch_dataset/2026/patches.zarr attrs= {}
Arrays: 0

ZARR FOLDER: modis/modis_patch_dataset
NOT FOUND

ZARR FOLDER: m

In [6]:
# Inventory plausible key columns. This is a clue for merge design, not a guarantee of uniqueness.
key_terms = ("sample_id", "station_id", "station_key", "network", "sample_date", "date", "latitude", "longitude",
             "point_id", "cluster_id", "scene_id", "year", "row", "col", "smap_x_idx", "smap_y_idx")
for item in report["parquet"]:
    names = [c["name"] for c in item["columns"]]
    matches = [n for n in names if any(t in n.lower() for t in key_terms)]
    report["key_inventory"].append({"path": item["path"], "rows": item["rows"], "candidate_key_columns": matches})
print("Candidate key inventory:")
display(pd.DataFrame(report["key_inventory"]))


Candidate key inventory:


,path,rows,candidate_key_columns
0,global_ismn_daily.parquet,5045849,"[station_id, network, network_name, latitude, ..."
1,station_date_candidates.parquet,3514643,"[sample_id, station_id, sample_date, network, ..."
2,stations.parquet,2210,"[station_id, network, network_name, latitude, ..."
3,era5_backend/era5_by_sample.parquet,3514643,"[sample_id, station_id, sample_date, era5_poin..."
4,era5_backend/era5_point_index.parquet,1489,[era5_point_id]
5,era5_backend/era5_sample_features.parquet,3514643,"[sample_id, station_id, sample_date, latitude,..."
6,smap/smap_by_sample.parquet,3514643,"[sample_id, station_id, sample_date, network, ..."
7,smap/station_smap_grid_mapping.parquet,2210,"[station_id, network, network_name, latitude, ..."
8,sentinel1/clusters.parquet,819,"[cluster_id, min_date, max_date]"
9,sentinel1/stations_with_clusters.parquet,2210,"[station_id, network, network_name, latitude, ..."


In [7]:
# Save portable reports to Data/Global.
REPORT_JSON.write_text(json.dumps(report, indent=2, default=safe), encoding="utf-8")
lines = ["GLOBAL MERGE INPUT INSPECTION", f"Created: {report['created_at']}", f"Root: {report['root']}", "", "PARQUET"]
for x in report["parquet"]:
    lines += [f"\n[{x['path']}] rows={x['rows']:,} row_groups={x['row_groups']} bytes={x['size_bytes']:,}",
              "Columns: " + ", ".join(f"{c['name']} ({c['type']})" for c in x["columns"])]
lines += ["", "CSV"]
for x in report["csv"]: lines += [f"\n[{x['path']}]", "Columns: " + ", ".join(x.get("columns", [])), "Error: " + x["error"] if x.get("error") else ""]
lines += ["", "ZARR"]
for x in report["zarr"]:
    lines += [f"\n[{x['path']}]", "Attributes: " + json.dumps(x.get("attrs", {}), default=safe)]
    if x.get("error"): lines.append("Error: " + x["error"])
    for a in x.get("arrays", []): lines.append(f"  {a['name']}: shape={a['shape']} dtype={a['dtype']} chunks={a['chunks']}")
lines += ["", "CANDIDATE KEY COLUMNS"]
for x in report["key_inventory"]: lines.append(f"{x['path']} ({x['rows']:,} rows): " + ", ".join(x["candidate_key_columns"]))
REPORT_TXT.write_text("\n".join(lines), encoding="utf-8")
print("\nSaved text report:", REPORT_TXT.resolve())
print("Saved JSON report:", REPORT_JSON.resolve())
print("Share either report for the next step: confirming join cardinality and building the merge pipeline.")
con.close()



Saved text report: /home/mayur/Code/UG_SM/Data/Global/global_merge_schema_report.txt
Saved JSON report: /home/mayur/Code/UG_SM/Data/Global/global_merge_schema_report.json
Share either report for the next step: confirming join cardinality and building the merge pipeline.
